# EDA: Compromisos Financieros (Carga Fija)
Este notebook calcula la carga financiera fija de los usuarios (mensualidades + cargos recurrentes) para estimar el `ratio_deuda_ingreso`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE_TXN = Path("Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")
BASE_CONV = Path("Datathon_Hey_dataset_conversaciones 1/dataset_conversaciones")

df_clientes = pd.read_csv(BASE_TXN / "hey_clientes.csv", dtype={"user_id": str})
df_productos = pd.read_csv(BASE_TXN / "hey_productos.csv", dtype={"producto_id": str, "user_id": str})
df_transacc = pd.read_csv(BASE_TXN / "hey_transacciones.csv", dtype={"transaccion_id": str, "user_id": str, "producto_id": str})
df_convs = pd.read_parquet(BASE_CONV / "dataset_50k_anonymized.parquet")


## 1. Mapeo de compromisos financieros (Carga Fija Total)

In [2]:
mensualidades = (df_productos[df_productos["monto_mensualidad"].notna()]
                 .groupby("user_id")["monto_mensualidad"].sum()
                 .rename("total_mensualidades").to_frame())

recurrentes = (df_transacc[df_transacc["tipo_operacion"] == "cargo_recurrente"]
               .groupby("user_id")["monto"].sum()
               .rename("total_recurrentes").to_frame())

carga = mensualidades.join(recurrentes, how="outer").fillna(0)
carga["carga_fija_total"] = carga["total_mensualidades"] + carga["total_recurrentes"]
carga = carga.reset_index()
carga.head()


,user_id,total_mensualidades,total_recurrentes,carga_fija_total
0,USR-00001,0.0,3891.0,3891.0
1,USR-00002,0.0,4688.0,4688.0
2,USR-00003,0.0,3542.0,3542.0
3,USR-00004,0.0,2195.0,2195.0
4,USR-00005,0.0,5583.0,5583.0


## 2. Ratio Deuda/Ingreso

In [3]:
df_analisis = df_clientes[["user_id", "ingreso_mensual_mxn"]].merge(carga, on="user_id", how="left")
df_analisis["carga_fija_total"] = df_analisis["carga_fija_total"].fillna(0)
df_analisis["ratio_deuda_ingreso"] = df_analisis["carga_fija_total"] / df_analisis["ingreso_mensual_mxn"]

deciles = df_analisis["ratio_deuda_ingreso"].quantile(np.arange(0.1, 1.1, 0.1))
print("Distribución del ratio_deuda_ingreso por decil:")
print(deciles)


Distribución del ratio_deuda_ingreso por decil:
0.1    0.000000
0.2    0.014452
0.3    0.028235
0.4    0.044506
0.5    0.066444
0.6    0.096456
0.7    0.142028
0.8    0.207728
0.9    0.316413
1.0    1.463298
Name: ratio_deuda_ingreso, dtype: float64


In [4]:
usuarios_riesgo = (df_analisis["ratio_deuda_ingreso"] > 0.4).sum()
total_usuarios = len(df_analisis)
pct_riesgo = (usuarios_riesgo / total_usuarios) * 100
print(f"\n% de usuarios con ratio > 0.4: {pct_riesgo:.2f}% ({usuarios_riesgo} usuarios de {total_usuarios})")



% de usuarios con ratio > 0.4: 5.82% (874 usuarios de 15025)


## 3. Top 5 categorías de cargos recurrentes

In [5]:
top_5_categorias = (df_transacc[df_transacc["tipo_operacion"]=="cargo_recurrente"]
                    .groupby("categoria_mcc")["monto"].sum()
                    .sort_values(ascending=False).head(5))
print("\nTop 5 categorías de cargos recurrentes por monto:")
print(top_5_categorias)



Top 5 categorías de cargos recurrentes por monto:
categoria_mcc
servicios_digitales    26448351.0
Name: monto, dtype: float64


## 4. Conversaciones sobre pagos automáticos y mensualidades

In [6]:
palabras_clave = "mensualidad|pago automático|suscripción"
convs_filtradas = df_convs[df_convs["input"].str.contains(palabras_clave, case=False, na=False) | 
                           df_convs["output"].str.contains(palabras_clave, case=False, na=False)]
print(f"\nNúmero de turnos de conversación sobre estos temas: {len(convs_filtradas)}")
convs_filtradas[["input", "output"]].head()



Número de turnos de conversación sobre estos temas: 1936


,input,output
7,"Hola buenos días , me podrías dar información ...",¡Claro que puedo ayudarte! Los plazos para el ...
8,A 18 y si se puede 24,"La mensualidad que tendrás que pagar, para un ..."
10,"Y por 20,000 a 24 meses de cuánto sería la men...","La mensualidad que tendrás que pagar, para un ..."
29,12 meses,"La mensualidad que tendrás que pagar, para un ..."
32,Ok,"Claro, me alegra que la información te haya qu..."
